# European Flight Delay Prediction — Train/Development/Test splits

This notebook takes the sample of 100,000 previously created from the Flight Delays 2023–2025 Parquet raw dataset from Hugging Face and stored in CGS bucket and creates separate parquet files for Training, Development and Test.

The sample dataset was created in /notebooks/02_analysis_and_downsampling.ipynb


For reproducability we are capturing and storing the following
* CGS generation version of sample dataset used - included in provenance.json uploaded with split files
* SHA-256 checksum finger prints for the sample dataset -  included in provenance.json uploaded with split files
* Split ID manifests - will be uploaded to GCS with split files
* Git commit version of this notebook use to create the splits - will be included in the run_id which is used for the GCS folder where the split files are uploaded


## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [ ]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [ ]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME

If you want to reproduct a previous run - you can get these details from the provenance.json file in the appropriate processed/splits/runs/* folder
*   SOURCE_GENERATION - update from `None` to a previously recorded generation value of the processed/flights_downsampled_100k.parquet from the run provenance.json input.generation value
* EXPECTED_SOURCE_SHA256 - If you supply SOURCE_GENERATION and want to verify it has expected SHA-256 fingerprint update from `None` to the run provenance.json input.sha256 value






In [ ]:
from google.colab import auth
auth.authenticate_user()

import hashlib
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import sklearn
from google.cloud import storage
from sklearn.model_selection import train_test_split, StratifiedKFold

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"
SOURCE_OBJECT = "processed/flights_downsampled_100k.parquet"
SOURCE_GENERATION = None
EXPECTED_SOURCE_SHA256 = None
RANDOM_STATE = 42
TARGET_COLUMN = "delayed_15min"
N_FOLDS = 8
UPLOAD_TO_GCS = True
PROCESSED_GCS_FOLDER = "processed/splits/runs/"

WORK_DIR = Path("/content/flight_split_run")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE = WORK_DIR / "flights_downsampled_100k.parquet"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

## 3. Download processed dataset
Capture and download the specific version of the processed dataset from CGS based on the SOURCE_GENERATION

In [ ]:
if SOURCE_GENERATION is None:
    latest_blob = bucket.blob(SOURCE_OBJECT) # Read latest version if no SOURCE_GENERATION supplied
    latest_blob.reload() # Get this objects metadata
    source_generation = int(latest_blob.generation) # Store its generation number as an integer
else:
    source_generation = int(SOURCE_GENERATION) # Else use generation number supplied

source_blob = bucket.blob(SOURCE_OBJECT, generation=source_generation) # Download the file based with the specific generation version value to local storage
source_blob.download_to_filename(str(LOCAL_SOURCE))

# Calculate a unique SHA-256 digital fingerprint for file at the path specified and return it as 64 character hexadecimal string
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_sha256 = sha256_file(LOCAL_SOURCE) # The SHA-256 digital fingerprint generated for the local file will later be saved in our provenance.json file
if EXPECTED_SOURCE_SHA256 is not None:
    assert source_sha256 == EXPECTED_SOURCE_SHA256, (
        "Downloaded file does not match the expected SHA-256."
    )

print("Source SHA-256:", source_sha256)
print("Input:", f"gs://{BUCKET_NAME}/{SOURCE_OBJECT}")
print("Pinned generation:", source_generation)
print("SHA-256:", source_sha256)
print("Downloaded to:", LOCAL_SOURCE)

## 4. Load and Validate the processed dataset
Check the following
* It has the correct number of rows
* There are no missing id's
* There are no duplicate id's
* There is no missing values in the TARGET_COLUMN i.e. delayed_15min column and it only contains values 0 or 1

In [ ]:
df = pd.read_parquet(LOCAL_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

assert len(df) == 100_000, f"Expected 100,000 rows, got {len(df):,}"
assert df["id"].notna().all(), "Missing flight IDs"
assert df["id"].is_unique, "Flight IDs are not unique"
assert df[TARGET_COLUMN].notna().all(), "Missing target values"
assert set(df[TARGET_COLUMN].unique()).issubset({0, 1}), "Unexpected target classes"
print("Input validation passed.")

## 5. Stratified 80/10/10 split

Stratifying on the TARGET_COLUM i.e `delayed_15min` preserves approximately the same delay rate in each dataset as in the original dataset.

The delay rate of each dataset output at the end should be similar.


Note: This is a random baseline, not a chronological or unseen-airport holdout. Those can be separate later experiments.

In [ ]:
# Split dataset into two groups - 80% in train_df and 20% in temp_df using RANDOM_STATE so can be reproduced if necessary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)

# Split the temp_df now into dev and test 50/50 so they will each end up with 10%
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df[TARGET_COLUMN],
)

print(f"Splits complete: Train ({len(train_df):,}), Dev ({len(dev_df):,}), Test ({len(test_df):,})")
print("Delay rates:")
display(pd.DataFrame({
    "dataset": ["full", "train", "dev", "test"],
    "rows": [len(df), len(train_df), len(dev_df), len(test_df)],
    "delay_rate_pct": [100*x[TARGET_COLUMN].mean() for x in (df, train_df, dev_df, test_df)] # Calculate the average value of the TARGET_COLUMN and multiply by 100 to get percentage
}).round(3))

## 6. Create 8 stratified cross-validation folds from TRAIN

Create 8 stratified cross validation folds from the training data with approximately 10,000 records each.

A model will then be trained with 7 of the folds and validated using the other which it wont yet have seen.

This provides us with the option to try 8 different models using the training data alone.

We save an ID-to-fold manifest so fold assignments can be audited without storing eight duplicate datasets



In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE) # Define fold strategy in this case
cv_folds = {}

# This will be used to store record of which flight Id's were assigned to each fold for reproducability and auditing, it will contain 8 tables with approximately 10,000 rows of validation records along with the fold number
cv_assignment_parts = []

# split the training data into the specified number of folds considering the TARGET_COLUMN, enumerate returns fold numbers from 0 to 7 in this case
# and loop through that number of times using different training/validation combination each time
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[TARGET_COLUMN])):
    # create two dataframes for the current fold (training and validation) selecting rows based on their numerical positions
    cv_folds[fold_idx] = (
        train_df.iloc[train_idx].copy(),
        train_df.iloc[val_idx].copy(),
    )

    # Extract the unique flight ID's for the 10,000 validation flights for this fold into a NumPy Array and stores these along with the fold number
    # in cv_assignment_parts
    cv_assignment_parts.append(pd.DataFrame({
        "id": train_df.iloc[val_idx]["id"].to_numpy(),
        "validation_fold": fold_idx,
    }))

# Combine the eight smaller cv_assignment_parts DataFrames into one larger DataFrame.
cv_manifest = pd.concat(cv_assignment_parts, ignore_index=True)

# Ensure that all flights from the training dataset have been assigned to a fold for validation and that all the id's are unique
assert len(cv_manifest) == len(train_df)
assert cv_manifest["id"].is_unique
print("Cross-validation folds:", len(cv_folds))
print(cv_manifest.groupby("validation_fold").size())

## 7. Verify no records are missing or duplicated between splits


In [ ]:
train_ids, dev_ids, test_ids = (set(x["id"]) for x in (train_df, dev_df, test_df))
assert (len(train_df), len(dev_df), len(test_df)) == (80_000, 10_000, 10_000)
assert train_ids.isdisjoint(dev_ids)
assert train_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)
assert train_ids | dev_ids | test_ids == set(df["id"])
print("All split checks passed.")

## 8. Capture the latest Git Commit

Record which version of the notebook that was used to create the splits. This will be used when generating the run_id in a later step which is used as the folder for the split files in GCS


In [ ]:
# Since we already did '%cd' into REPO_DIR, the current directory '.' is our git repository.
git_target_dir = "." if os.path.basename(os.getcwd()) == REPO_DIR else REPO_DIR

# Use subprocess to run command outside of python and capture result - in this case get latest commit on HEAD
git_commit = subprocess.check_output(
    ["git", "-C", str(git_target_dir), "rev-parse", "HEAD"], text=True
).strip()

# Check for changes which have not yet been committed
git_status = subprocess.check_output(
    ["git", "-C", str(git_target_dir), "status", "--porcelain"], text=True
).strip()

print("Git commit:", git_commit)
print("Working tree clean:", not bool(git_status))
if git_status:
    print("WARNING: Uncommitted changes exist, commit them for full code reproducibility.")

## 9. Save Parquet splits and CSV manifests locally


In [ ]:
# Create dictionary (key/value) with 3 entries where key is the datatype and value is the previously created dataframe
splits = {"train": train_df, "dev": dev_df, "test": test_df}

# Save each dataframe train,dev and test as parquet file, index false means we will retain flight id as the id
for name, frame in splits.items():
    frame.to_parquet(WORK_DIR / f"flights_{name}.parquet", index=False, engine="pyarrow")

# Create dataframe with two columns, the id and which split its part of and save it locally
split_manifest = pd.concat([
    frame[["id"]].assign(split=name)
    for name, frame in splits.items()
], ignore_index=True).sort_values("id")

split_manifest.to_csv(WORK_DIR / "split_manifest.csv", index=False)
cv_manifest.sort_values("id").to_csv(WORK_DIR / "cv_fold_manifest.csv", index=False)

output_files = [WORK_DIR / f"flights_{name}.parquet" for name in splits]
output_files += [WORK_DIR / "split_manifest.csv", WORK_DIR / "cv_fold_manifest.csv"]
print("Saved:", [p.name for p in output_files])

## 10. Upload the split files including the manifests to run specific GCS folder

Note the run_id is a combination of the current timestamp and the first 12 characters of the git_commit.

Ensure you set UPLOAD_TO_GCS = True in Step 2 if you want the files to be uploaded to GCS

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id = f"{timestamp}-{git_commit[:12]}"
output_prefix = f"{PROCESSED_GCS_FOLDER}{run_id}"

receipts = {}
if UPLOAD_TO_GCS:
    for path in output_files:
        object_name = f"{output_prefix}/{path.name}"
        blob = bucket.blob(object_name) # Create python object representing the GCS destination
        blob.upload_from_filename(str(path), if_generation_match=0) # Upload - fail if object already exists with same name
        # Record info about the file we have uploaded, generation and sha256 are based on the specific split file we are uploading
        receipts[path.name] = {
            "gcs_uri": f"gs://{BUCKET_NAME}/{object_name}",
            "generation": str(blob.generation),
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }
        print(path.name, "→ generation", blob.generation)
else:
    print("Upload disabled. Local outputs were saved only.")

## 11. Create and upload provenance.json file to the same run folder in GCS

Create provenance json file with all necessary details on the source file used and spilt files created and upload to GCS is UPLOAD_TO_GCS is True

In [ ]:
provenance = {
    "run_id": run_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "processing_notebook": "notebooks/03_data_splitting.ipynb",
    "git_commit": git_commit,
    "git_working_tree_clean": not bool(git_status),
    "input": {
        "gcs_uri": f"gs://{BUCKET_NAME}/{SOURCE_OBJECT}",
        "generation": str(source_generation),
        "sha256": source_sha256,
        "row_count": len(df),
    },
    "method": {
        "split_type": "stratified random 80/10/10",
        "target_column": TARGET_COLUMN,
        "random_seed": RANDOM_STATE,
        "cv_type": "StratifiedKFold",
        "cv_folds": N_FOLDS,
    },
    "split_rows": {name: len(frame) for name, frame in splits.items()},
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pa.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "outputs": receipts,
}

provenance_path = WORK_DIR / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")

if UPLOAD_TO_GCS:
    provenance_object = f"{output_prefix}/provenance.json"
    provenance_blob = bucket.blob(provenance_object)
    provenance_blob.upload_from_filename(
        str(provenance_path), content_type="application/json", if_generation_match=0
    )
    print("Provenance:", f"gs://{BUCKET_NAME}/{provenance_object}")
    print("Provenance GCS generation:", provenance_blob.generation)
else:
    print("Local provenance:", provenance_path)

display(pd.DataFrame([{
    "file": name, "generation": receipt["generation"], "sha256": receipt["sha256"]
} for name, receipt in receipts.items()]))


## 12. Verify files uploaded correctly

Verify uploaded files match the receipts we stored



In [ ]:
if UPLOAD_TO_GCS:
    for path in output_files:
        receipt = receipts[path.name]
        object_name = f"{output_prefix}/{path.name}"
        uploaded = bucket.blob(object_name, generation=int(receipt["generation"])) # Find the file from GCS which matches our receipt generation
        uploaded.reload() # Get the objects metadata without downloading it
        assert str(uploaded.generation) == receipt["generation"]
        assert int(uploaded.size) == path.stat().st_size
    print("All uploaded file generations and sizes verified.")
